In [1]:
from langchain_community.tools import tool
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import HumanMessage
import requests
import os
from dotenv import load_dotenv
from langchain_core.tools import InjectedToolArg
from typing import Annotated
load_dotenv()

/tmp/ipykernel_7352/511356723.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.tools import tool


True

In [10]:
### Conversionn Rate Tool



@tool(parse_docstring=True)
def get_conversion_rate(base_currency: str, target_currency: str) -> float:
    """Fetch the current exchange rate between two currencies.

    Use this tool whenever you need the live exchange rate from one currency to
    another, either because the user asked for the rate directly, or because you
    need a rate before converting an amount. Never guess or recall
    exchange rates from memory, because they change constantly.

    Args:
        base_currency: The 3-letter ISO 4217 code of the currency being converted FROM, in uppercase, e.g. "USD", "EUR", "PKR".
        target_currency: The 3-letter ISO 4217 code of the currency being converted TO, in uppercase, e.g. "GBP", "JPY", "PKR".

    Returns:
        The Conversion rate between two currencies
    """
    api_key = os.getenv("EXCHANGE_API_KEY")
    url = f"https://v6.exchangerate-api.com/v6/{api_key}/pair/{base_currency.upper()}/{target_currency.upper()}"

    response = requests.get(url)
    data = response.json()

    if data.get("result") != "success":
        raise ValueError(f"API error: {data.get('error-type', data)}")

    return data["conversion_rate"]





### Currency Convertor Tool
@tool(parse_docstring=True)
def convert_currency(
    base_currency_value: float,
    conversion_rate: Annotated[float, InjectedToolArg],
) -> float:
    """Convert an amount of money from one currency to another.

    Call this tool together with `get_conversion_rate` whenever the user wants
    to convert an amount. The exchange rate is injected automatically by the
    application, so you only need to provide the amount. Do not wait for the
    rate result before calling this tool.

    Args:
        base_currency_value: The amount of money in the base currency, e.g. 250.0 for 250 EUR.
        conversion_rate: Injected by the application. Do not provide this.
    """
    return base_currency_value * conversion_rate

convert_currency.invoke({"base_currency_value":1,"conversion_rate":get_conversion_rate.invoke({"base_currency":"USD","target_currency":"PKR"})})

277.0498

In [11]:
### Tools Attaching to LLM

tools={"get_conversion_rate":get_conversion_rate
       ,"convert_currency":convert_currency}

primary_llm = ChatGoogleGenerativeAI(
    model="gemini-3.1-flash-lite",
    temperature=0.7,
 
)

backup_llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",   
    temperature=0.7,
)

llm_with_tools = primary_llm.bind_tools([get_conversion_rate,convert_currency]).with_fallbacks(
    [backup_llm.bind_tools([get_conversion_rate,convert_currency])]
)

In [12]:
### Creating History List
messages=[HumanMessage("Convert the 56 USD to PkR")]

In [13]:
llm_with_tools.invoke(messages)

AIMessage(content=[], additional_kwargs={'function_call': {'name': 'convert_currency', 'arguments': '{"base_currency_value": 56}'}, '__gemini_function_call_thought_signatures__': {'call_346813': 'EnMKcQFpFH0TTkh1EJtIIrrmP4QJ0sY9gnyXSbB2o8NwgiDAOng0kCfLXh4cndhK2IwVr6gi791xBdSRDVz+4Z7ula4dQe88Z0SFdHrd/9PbTNhbLRK6rqem1as6q4D5M8LqRVXl1JFQhkGQ+uURkwdDH/Z5'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a1667a-1d5e-7ff3-bafe-82e666488042-0', tool_calls=[{'name': 'get_conversion_rate', 'args': {'target_currency': 'PKR', 'base_currency': 'USD'}, 'id': 'call_346813', 'type': 'tool_call'}, {'name': 'convert_currency', 'args': {'base_currency_value': 56}, 'id': 'call_346814', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 357, 'output_tokens': 48, 'total_tokens': 405, 'input_token_details': {'cache_read': 0}})